## Tasks 1 and 2(Ingestion and Cleaning)
Task 01: Write explicit StructType schemas for all 4 CSVs. Load them with header=True but no inferSchema. Any row that fails casting goes into a "rejected Dataframe which you log separately.

In [105]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
spark = SparkSession.builder.getOrCreate()

In [145]:
customerStruct = StructType([
    StructField('customer_id', StringType(), True),
    StructField('customer_name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('country', StringType(), True),
    StructField('customer_tier', StringType(), True),
    StructField('signup_date', StringType(), True),
])

In [266]:
df_customers = spark.read.format('csv')\
                .schema(customerStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/customers.csv')

In [ ]:
val = [df_customers.collect()[0]["signup_date"]]
print(val)
formatted_date = []

# Check if the date value is not None before processing
if val[0] is not None and '/' in val[0]:
    year = val[0][6:10]
    month = val[0][3:5]
    day = val[0][0:2]
    new_date = f'{year}-{month}-{day}'
    print(new_date)
    formatted_date.append(new_date)
else:
    # Handle the case when date is None or doesn't contain '/'
    if val[0] is not None:
        formatted_date.append(val[0])  # Keep original format if no '/'
    else:
        formatted_date.append("No date available")  # Handle None case

print(formatted_date)

In [186]:
df_customers.show()

+-----------+-----------------+--------------------+-------+-------------+-----------+
|customer_id|    customer_name|               email|country|customer_tier|signup_date|
+-----------+-----------------+--------------------+-------+-------------+-----------+
|     C00001|    Donald Walker|rhodespatricia@ex...|     UA|         gold| 12/12/2018|
|     C00002|     Brandon Hall|hoffmanjennifer@e...|     ST|       BRONZE| 2021-08-03|
|     C00003|    Kevin Pacheco|blakeerik@example...|     LV|       Bronze| 27/07/2021|
|     C00004|     Tyler Rogers|jamesmichael@exam...|     UY|       bronze| 13/05/2020|
|     C00005|   Monica Herrera| smiller@example.net|     VN|         gold| 2020-04-01|
|     C00006| Michele Williams|kendragalloway@ex...|     BZ|        Gold | 28/01/2021|
|     C00007|        Mark Diaz|michael41@example...|     ZW|       bronze| 27/03/2021|
|     C00008|    Danielle Ford|veronica83@exampl...|     LR|       Bronze| 16/06/2019|
|     C00009|   Andrew Stewart|  carl95@exa

In [124]:
from pyspark.sql import functions as F

# Try formats in order of priority; unparsed rows become null
df_customers = df_customers.withColumn(
    "formatted_date",
    F.coalesce(
        F.to_date("signup_date", "yyyy-MM-dd"),
        F.to_date("signup_date", "MM/dd/yyyy"),
        F.to_date("signup_date", "yyyy/MM/dd")
    )
)


In [149]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)



In [150]:
orderItemStruct = StructType([
    StructField('item_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('product_name', StringType(), True),
    StructField('category', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('unit_price', FloatType(), True),
])

In [151]:
df_order_items = spark.read.format('csv')\
                .schema(orderItemStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/order_items.csv')

In [152]:
df_order_items.show()

+--------+--------+-------------------+-------------+--------+----------+
| item_id|order_id|       product_name|     category|quantity|unit_price|
+--------+--------+-------------------+-------------+--------+----------+
|I0000001| O000001|         Itself Cup|         Toys|       6|      8.77|
|I0000002| O000001|  Tonight Authority|         Toys|       5|    465.98|
|I0000003| O000001|      Court Century|       Sports|       2|    304.07|
|I0000004| O000001|       Figure Bring|       Beauty|       7|    158.51|
|I0000005| O000002|        Public Lose|Home & Garden|       3|    387.15|
|I0000006| O000002|      Prepare Local|  Electronics|       8|    487.97|
|I0000007| O000002|      Expert School|       Sports|       8|    476.36|
|I0000008| O000002|    Prevent Million|       Sports|       1|    198.84|
|I0000009| O000002|           Cup Full|Home & Garden|      10|    274.63|
|I0000010| O000003|        After Civil|        Books|       1|    182.89|
|I0000011| O000003|         Appear Nor

In [153]:
df_order_items.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: float (nullable = true)



In [154]:
orderStruct = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", FloatType(), True),
    StructField("discount_pct",IntegerType(), True)
    ])

In [155]:
df_orders = spark.read.format('csv')\
                .schema(orderStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/orders.csv')
df_orders.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
| O000001|     C00078|03/08/2022|cancelled|      230.07|           5|
| O000002|     C00044|22/01/2021|completed|      221.77|           5|
| O000003|     C00286|2024-10-17|  pending|      1795.1|          15|
| O000004|     C00398|16/05/2022| refunded|      404.43|          20|
| O000005|     C00231|21/03/2024|cancelled|      -26.78|          15|
| O000006|     C00302|18/07/2024|cancelled|      123.89|          15|
| O000007|     C00292|2023-04-05|completed|     1677.58|          20|
| O000008|     C00313|24/10/2021|completed|     1492.92|          25|
| O000009|     C00391|2021-01-28|  pending|      394.15|           5|
| O000010|     C00136|18/11/2023|completed|      118.46|          25|
| O000011|     C00081|2021-01-07|  pending|      354.52|           5|
| O000012|     C0028

In [156]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- discount_pct: integer (nullable = true)



In [157]:
returnStruct = StructType ([
    StructField('return_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('return_date', StringType(), True),
    StructField('reason', StringType(), True),
    StructField('refund_amount', FloatType(), True)
])

In [158]:
df_returns = spark.read.format('csv')\
                .schema(returnStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/returns.csv')
df_returns.show()

+---------+--------+-----------+-------------------+-------------+
|return_id|order_id|return_date|             reason|refund_amount|
+---------+--------+-----------+-------------------+-------------+
|  R000001| O001713| 19/04/2024|       changed mind|       349.04|
|  R000002| O001177| 05/08/2022|         wrong item|      1721.96|
|  R000003| O000991| 2021-01-08|   not as described|        27.77|
|  R000004| O000017| 12/01/2023|          defective|       260.67|
|  R000005| O001787| 2021-09-05|          defective|       754.97|
|  R000006| O000732| 20/06/2022|          defective|       240.01|
|  R000007| O001191| 05/06/2021|   not as described|       913.52|
|  R000008| O000666| 2023-01-02|       changed mind|        27.48|
|  R000009| O000147| 2023-05-05|          defective|         71.9|
|  R000010| O000746| 2023-12-03|         wrong item|       165.16|
|  R000011| O001509| 2022-09-05|         wrong item|       492.46|
|  R000012| O001431| 22/12/2023|damaged in shipping|      1730

In [159]:
df_returns.printSchema()

root
 |-- return_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- return_date: string (nullable = true)
 |-- reason: string (nullable = true)
 |-- refund_amount: float (nullable = true)



Task 02: Clean in the exact sequence the spec says deduplicate-> normalize dates -> lowercase tier -> drop nulls -> flag negatives. Handling the mixed date formats(year-month-day vs day-month-year). use F.to/-date with coalesce to try both formats.

In [170]:
# Remove exact duplicate rows using dropDuplicates().
df_customers = df_customers.dropDuplicates()
df_order_items = df_order_items.dropDuplicates()
df_returns = df_returns.dropDuplicates()
df_orders = df_orders.dropDuplicates()

In [187]:
df_customers = df_customers.withColumn(
    'signup_date',
    coalesce(
        try_to_date(col("signup_date"), "yyyy-MM-dd"),
        try_to_date(col("signup_date"), "dd/MM/yyyy")
    )
)

In [188]:
df_customers.show()

+-----------+-----------------+--------------------+-------+-------------+-----------+
|customer_id|    customer_name|               email|country|customer_tier|signup_date|
+-----------+-----------------+--------------------+-------+-------------+-----------+
|     C00001|    Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12|
|     C00002|     Brandon Hall|hoffmanjennifer@e...|     ST|       BRONZE| 2021-08-03|
|     C00003|    Kevin Pacheco|blakeerik@example...|     LV|       Bronze| 2021-07-27|
|     C00004|     Tyler Rogers|jamesmichael@exam...|     UY|       bronze| 2020-05-13|
|     C00005|   Monica Herrera| smiller@example.net|     VN|         gold| 2020-04-01|
|     C00006| Michele Williams|kendragalloway@ex...|     BZ|        Gold | 2021-01-28|
|     C00007|        Mark Diaz|michael41@example...|     ZW|       bronze| 2021-03-27|
|     C00008|    Danielle Ford|veronica83@exampl...|     LR|       Bronze| 2019-06-16|
|     C00009|   Andrew Stewart|  carl95@exa

In [195]:
df_orders = df_orders.withColumn(
    'order_date',
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy")
    )
  )
df_orders.show()

+--------+-----------+----------+---------+------------+------------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+---------+------------+------------+------------------+
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|
| O000075|     C00270|2023-04-11|completed|     1142.73|          15|             False|
| O000284|     C00110|2024-09-18| refunded|       157.4|          10|             False|
| O001047|     C00023|2024-04-14| refunded|      955.24|          10|             False|
| O001768|     C00217|2022-10-02|cancelled|     1549.06|          10|             False|
| O001817|     C00163|2022-05-02|completed|     1951.94|           0|             False|
| O001657|     C00342|2024-04-20|completed|     1730.68|           5|             False|
| O001945|     C00098|2022-03-22| refunded|       354.3|          10|             False|
| O000057|     C00213

In [196]:
## Standardise customer_tier to lowercase.
df_customers = df_customers.withColumn("customer_tier", lower("customer_tier"))
df_customers.show()

+-----------+-----------------+--------------------+-------+-------------+-----------+
|customer_id|    customer_name|               email|country|customer_tier|signup_date|
+-----------+-----------------+--------------------+-------+-------------+-----------+
|     C00001|    Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12|
|     C00002|     Brandon Hall|hoffmanjennifer@e...|     ST|       bronze| 2021-08-03|
|     C00003|    Kevin Pacheco|blakeerik@example...|     LV|       bronze| 2021-07-27|
|     C00004|     Tyler Rogers|jamesmichael@exam...|     UY|       bronze| 2020-05-13|
|     C00005|   Monica Herrera| smiller@example.net|     VN|         gold| 2020-04-01|
|     C00006| Michele Williams|kendragalloway@ex...|     BZ|        gold | 2021-01-28|
|     C00007|        Mark Diaz|michael41@example...|     ZW|       bronze| 2021-03-27|
|     C00008|    Danielle Ford|veronica83@exampl...|     LR|       bronze| 2019-06-16|
|     C00009|   Andrew Stewart|  carl95@exa

In [197]:
## Drop rows where order_id or customer_id is NULL.
df_customers = df_customers.dropna(subset=["customer_id"])
df_order_items = df_order_items.dropna(subset=["order_id"])
df_returns = df_returns.dropna(subset=["order_id"])
df_orders = df_orders.dropna(subset=["order_id", "customer_id"])
df_customers.show()

+-----------+-----------------+--------------------+-------+-------------+-----------+
|customer_id|    customer_name|               email|country|customer_tier|signup_date|
+-----------+-----------------+--------------------+-------+-------------+-----------+
|     C00001|    Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12|
|     C00002|     Brandon Hall|hoffmanjennifer@e...|     ST|       bronze| 2021-08-03|
|     C00003|    Kevin Pacheco|blakeerik@example...|     LV|       bronze| 2021-07-27|
|     C00004|     Tyler Rogers|jamesmichael@exam...|     UY|       bronze| 2020-05-13|
|     C00005|   Monica Herrera| smiller@example.net|     VN|         gold| 2020-04-01|
|     C00006| Michele Williams|kendragalloway@ex...|     BZ|        gold | 2021-01-28|
|     C00007|        Mark Diaz|michael41@example...|     ZW|       bronze| 2021-03-27|
|     C00008|    Danielle Ford|veronica83@exampl...|     LR|       bronze| 2019-06-16|
|     C00009|   Andrew Stewart|  carl95@exa

In [198]:
## Add a boolean column is_negative_amount to flag (not drop) orders with total_amount < 0.
df_orders = df_orders.withColumn("is_negative_amount", when((col('total_amount') < 0), "True").otherwise("False"))
df_orders.show()


+--------+-----------+----------+---------+------------+------------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+---------+------------+------------+------------------+
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|
| O000075|     C00270|2023-04-11|completed|     1142.73|          15|             False|
| O000284|     C00110|2024-09-18| refunded|       157.4|          10|             False|
| O001047|     C00023|2024-04-14| refunded|      955.24|          10|             False|
| O001768|     C00217|2022-10-02|cancelled|     1549.06|          10|             False|
| O001817|     C00163|2022-05-02|completed|     1951.94|           0|             False|
| O001657|     C00342|2024-04-20|completed|     1730.68|           5|             False|
| O001945|     C00098|2022-03-22| refunded|       354.3|          10|             False|
| O000057|     C00213

## Task 03 and 04
### Joins + Aggregations

Task 03: Three joins - orders-> customers(inner), orders->order_items(left), then an anti-join to catch orphaned itens. Derive net_amount as a new column

In [201]:
df_order_customer= df_orders.join(df_customers, on=["customer_id"], how="inner")


In [202]:
df_order_customer.show()

+-----------+--------+----------+---------+------------+------------+------------------+-----------------+--------------------+-------+-------------+-----------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount|    customer_name|               email|country|customer_tier|signup_date|
+-----------+--------+----------+---------+------------+------------+------------------+-----------------+--------------------+-------+-------------+-----------+
|     C00222| O000028|2024-07-03|cancelled|      340.15|          25|             False|      Paul Finley|brandi90@example.com|     TD|       silver| 2021-04-06|
|     C00270| O000075|2023-04-11|completed|     1142.73|          15|             False|    Jeffrey Colon|kevinmorrison@exa...|     FR|      silver | 2020-10-20|
|     C00110| O000284|2024-09-18| refunded|       157.4|          10|             False|      Amanda Diaz|christopherphilli...|     NI|       bronze| 2020-08-17|
|     C00023| O001047|2024-0

In [203]:
df_orders_order_items = df_orders.join(df_order_items, on=["order_id"], how="left")
df_orders_order_items.show()

+--------+-----------+----------+---------+------------+------------+------------------+--------+---------------+-------------+--------+----------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount| item_id|   product_name|     category|quantity|unit_price|
+--------+-----------+----------+---------+------------+------------+------------------+--------+---------------+-------------+--------+----------+
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000093| Left Operation|         Toys|       9|     319.4|
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000090|  Center Enough|         Toys|       9|     487.2|
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000094|Financial Occur|        Books|       3|     68.68|
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000091|      Way Among

In [204]:
df_orphaned_order_items = df_orders.join(df_order_items, on=["order_id"], how="anti")
df_orphaned_order_items.show()

+--------+-----------+----------+------+------------+------------+------------------+
|order_id|customer_id|order_date|status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+------+------------+------------+------------------+
+--------+-----------+----------+------+------------+------------+------------------+



In [205]:
df_orphaned_customers = df_orders.join(df_customers, on=["customer_id"], how="anti")
df_orphaned_customers.show()

+-----------+--------+----------+------+------------+------------+------------------+
|customer_id|order_id|order_date|status|total_amount|discount_pct|is_negative_amount|
+-----------+--------+----------+------+------------+------------+------------------+
+-----------+--------+----------+------+------------+------------+------------------+



In [206]:
# Add a derived column net_amount = total_amount × (1 − discount_pct / 100)
df_order_customer = df_order_customer.withColumn('net_amount', (col("total_amount") * (1 - col("discount_pct")/100)))
df_order_customer.show()

+-----------+--------+----------+---------+------------+------------+------------------+-----------------+--------------------+-------+-------------+-----------+------------------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount|    customer_name|               email|country|customer_tier|signup_date|        net_amount|
+-----------+--------+----------+---------+------------+------------+------------------+-----------------+--------------------+-------+-------------+-----------+------------------+
|     C00222| O000028|2024-07-03|cancelled|      340.15|          25|             False|      Paul Finley|brandi90@example.com|     TD|       silver| 2021-04-06|255.11249542236328|
|     C00270| O000075|2023-04-11|completed|     1142.73|          15|             False|    Jeffrey Colon|kevinmorrison@exa...|     FR|      silver | 2020-10-20| 971.3204833984374|
|     C00110| O000284|2024-09-18| refunded|       157.4|          10|             False|      A

### Task 4 - Three window function outputs. partitionBy and orderBy for each - rank by country, rolling 7-day count, monthly revenue share per category. 

In [221]:
#  Customers ranked by lifetime net spend within each country.
## columns to use - net_amount, country, customer_id

from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [222]:
windowSpec = Window.partitionBy("customer_name").orderBy(F.desc("net_amount"))



In [232]:
# 1. Aggregate to calculate total lifetime spend per customer per country
df_lifetime = df_order_customer.groupBy("country", "customer_id", "customer_name") \
                .agg(F.sum("net_amount").alias("lifetime_net_spend"))

# 2. Define the Window to rank by spend within each country
# Highest spend gets rank 1
window_spec = Window.partitionBy("country").orderBy(F.col("lifetime_net_spend").desc())

# 3. Apply dense_rank() to handle potential spending ties gracefully
df_ranked = df_lifetime.withColumn("spend_rank", F.dense_rank().over(window_spec))

# Display the final ranking
df_ranked.orderBy("country", "spend_rank").show()

+-------+-----------+-------------------+------------------+----------+
|country|customer_id|      customer_name|lifetime_net_spend|spend_rank|
+-------+-----------+-------------------+------------------+----------+
|     AD|     C00090|       Brandon King|  10061.3364944458|         1|
|     AD|     C00047|      Jason Bentley| 9394.521502685546|         2|
|     AD|     C00300|      Melissa Wiley| 2176.942980957031|         3|
|     AD|     C00358|       Derek Wright| 105.8395034790039|         4|
|     AE|     C00331|        David Bruce| 9666.087084960938|         1|
|     AE|     C00309|      Justin Bishop|  6167.95106048584|         2|
|     AE|     C00312|     Melissa Abbott|2899.2920364379884|         3|
|     AE|     C00178|      William Gould|1725.6214569091799|         4|
|     AE|     C00010|       Thomas Ellis| 592.8674926757812|         5|
|     AG|     C00385|        Debra Ortiz| 3740.955453491211|         1|
|     AL|     C00122|     William Huerta| 20481.26496238709|    

In [230]:
# A 7-day rolling order count per customer (based on order_date).

df_oc = df_order_customer.withColumn("date_in_seconds", F.col("order_date").cast("timestamp").cast("long"))

seconds_in_7_days = 6 * 24 * 60 * 60 

window_spec = Window.partitionBy("customer_id") \
                    .orderBy("date_in_seconds") \
                    .rangeBetween(-seconds_in_7_days, Window.currentRow)

df_rolling = df_oc.withColumn(
    "rolling_7day_order_count", 
    F.count("order_id").over(window_spec)
).drop("date_in_seconds")

df_rolling.orderBy("customer_id", "order_date").show()

+-----------+--------+----------+---------+------------+------------+------------------+--------------+--------------------+-------+-------------+-----------+------------------+------------------------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount| customer_name|               email|country|customer_tier|signup_date|        net_amount|rolling_7day_order_count|
+-----------+--------+----------+---------+------------+------------+------------------+--------------+--------------------+-------+-------------+-----------+------------------+------------------------+
|     C00001| O000989|2022-05-31|  pending|      798.16|          10|             False| Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12| 718.3439758300782|                       1|
|     C00001| O001685|2022-06-17| refunded|      848.71|          25|             False| Donald Walker|rhodespatricia@ex...|     UA|         gold| 2018-12-12| 636.5325164794922|           

In [237]:
# Each product category’s share of total revenue per calendar month.
df_prod_cat = df_orders_order_items.groupBy("category", "order_date").agg(F.sum("total_amount").alias("total_revenue"))

# 3. Define a Window partitioned ONLY by month to sum up all categories together
monthly_window = Window.partitionBy("order_date")

# 4. Calculate the total monthly revenue and divide the category share
df_share = df_prod_cat \
    .withColumn("total_monthly_revenue", F.sum("total_revenue").over(monthly_window)) \
    .withColumn("revenue_share_pct", F.round((F.col("total_revenue") / F.col("total_monthly_revenue")) * 100, 2))

# Display the result sorted chronologically
df_share.orderBy("order_date", F.col("revenue_share_pct").desc()).show()

+-------------+----------+------------------+---------------------+-----------------+
|     category|order_date|     total_revenue|total_monthly_revenue|revenue_share_pct|
+-------------+----------+------------------+---------------------+-----------------+
|        Books|2021-01-01|1058.4300231933594|   3899.7901306152344|            27.14|
|       Sports|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|     Clothing|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|Home & Garden|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|         Toys|2021-01-01| 710.3400268554688|   3899.7901306152344|            18.21|
|     Clothing|2021-01-02| 1954.449951171875|    1954.449951171875|            100.0|
|         Toys|2021-01-04| 2095.340087890625|   5238.3502197265625|             40.0|
|       Sports|2021-01-04|1047.6700439453125|   5238.3502197265625|             20.0|
|        Books|2021-01-04|1047.6700439453125|   5238.3

In [238]:
## Join returns to your enriched orders DataFrame
df_order_customer = df_order_customer.join(df_ranked, on=["customer_id"], how="inner")
df_order_customer.show(10)

+-----------+--------+----------+---------+------------+------------+------------------+-----------------+--------------------+-------+-------------+-----------+------------------+-------+-----------------+------------------+----------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount|    customer_name|               email|country|customer_tier|signup_date|        net_amount|country|    customer_name|lifetime_net_spend|spend_rank|
+-----------+--------+----------+---------+------------+------------+------------------+-----------------+--------------------+-------+-------------+-----------+------------------+-------+-----------------+------------------+----------+
|     C00169| O000702|2022-10-09|cancelled|     1973.92|          15|             False|    Joseph Romero|myerstheodore@exa...|     MW|         gold| 2021-01-25|1677.8320373535155|     MW|    Joseph Romero|3393.5376037597657|         2|
|     C00126| O000875|2023-09-12| refunded|        8

In [240]:
df_orders_order_items.show(5)

+--------+-----------+----------+---------+------------+------------+------------------+--------+---------------+-------------+--------+----------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount| item_id|   product_name|     category|quantity|unit_price|
+--------+-----------+----------+---------+------------+------------+------------------+--------+---------------+-------------+--------+----------+
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000093| Left Operation|         Toys|       9|     319.4|
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000090|  Center Enough|         Toys|       9|     487.2|
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000094|Financial Occur|        Books|       3|     68.68|
| O000028|     C00222|2024-07-03|cancelled|      340.15|          25|             False|I0000091|      Way Among

In [244]:
## Compute return rates using groupBy aggregations
df_return_rates = df_orders_order_items.groupBy("category").agg(
    # 1. Total orders in this category
    F.count("order_id").alias("total_orders"),

    # 2. Returned orders — no .otherwise() so non-matches become NULL and count skips them
    F.count(
        F.when(F.col("status") == "refunded", F.col("order_id"))
    ).alias("total_returned")

).withColumn(
    # 3. Return rate percentage
    "return_rate_pct",
    F.round((F.col("total_returned") / F.col("total_orders")) * 100, 2)
)

df_return_rates.orderBy(F.col("return_rate_pct").desc()).show()

+-------------+------------+--------------+---------------+
|     category|total_orders|total_returned|return_rate_pct|
+-------------+------------+--------------+---------------+
|Home & Garden|         880|           223|          25.34|
|     Clothing|         820|           207|          25.24|
|       Sports|         761|           192|          25.23|
|        Books|         778|           194|          24.94|
|       Beauty|         800|           191|          23.88|
|  Electronics|         836|           197|          23.56|
|         Toys|         887|           200|          22.55|
+-------------+------------+--------------+---------------+



In [ ]:
## Flag where refund_amount > net_amount


In [ ]:
## Get top 10 refund customers


In [251]:
df_order_customer = df_order_customer.drop("country", "customer_name")  # drop one of the duplicates

In [264]:
df_order_customer.write \
    .mode('overwrite') \
    .parquet('C:/Users/ADMIN/Desktop/Assignment_1/')
    

Py4JJavaError: An error occurred while calling o1337.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

In [254]:
## writing summary tables to csv
df_return_rates.write('csv')\
    .mode('overwrite')\
    .option('path', 'C:/Users/ADMIN/Desktop/Assignment_1/output/return_rates.csv')\
    .save()

TypeError: 'DataFrameWriter' object is not callable

In [ ]:
## writing summary tables to csv
df_share.write('csv')\
    .mode('overwrite')\
    .option('path', 'C:\Users\ADMIN\Desktop\Lux Dev Data Enginering\Assignment_1\pc_share.csv')\
    .save()


This would prevent the TypeError by first checking if `val[0]` is not None before attempting to search for the '/' character within it.


Would you like me to provide the corrected code?

In [78]:
df_customers.show()

+-----------+-----------------+--------------------+-------+-------------+-----------+
|customer_id|    customer_name|               email|country|customer_tier|signup_date|
+-----------+-----------------+--------------------+-------+-------------+-----------+
|     C00001|    Donald Walker|rhodespatricia@ex...|     UA|         gold| 12/12/2018|
|     C00002|     Brandon Hall|hoffmanjennifer@e...|     ST|       BRONZE| 2021-08-03|
|     C00003|    Kevin Pacheco|blakeerik@example...|     LV|       Bronze| 27/07/2021|
|     C00004|     Tyler Rogers|jamesmichael@exam...|     UY|       bronze| 13/05/2020|
|     C00005|   Monica Herrera| smiller@example.net|     VN|         gold| 2020-04-01|
|     C00006| Michele Williams|kendragalloway@ex...|     BZ|        Gold | 28/01/2021|
|     C00007|        Mark Diaz|michael41@example...|     ZW|       bronze| 27/03/2021|
|     C00008|    Danielle Ford|veronica83@exampl...|     LR|       Bronze| 16/06/2019|
|     C00009|   Andrew Stewart|  carl95@exa

In [53]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)



In [30]:
orderItemStruct = StructType([
    StructField('item_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('product_name', StringType(), True),
    StructField('category', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('unit_price', FloatType(), True),
])

In [31]:
df_order_items = spark.read.format('csv')\
                .schema(orderItemStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/order_items.csv')

In [32]:
df_order_items.show()

+--------+--------+-------------------+-------------+--------+----------+
| item_id|order_id|       product_name|     category|quantity|unit_price|
+--------+--------+-------------------+-------------+--------+----------+
|I0000001| O000001|         Itself Cup|         Toys|       6|      8.77|
|I0000002| O000001|  Tonight Authority|         Toys|       5|    465.98|
|I0000003| O000001|      Court Century|       Sports|       2|    304.07|
|I0000004| O000001|       Figure Bring|       Beauty|       7|    158.51|
|I0000005| O000002|        Public Lose|Home & Garden|       3|    387.15|
|I0000006| O000002|      Prepare Local|  Electronics|       8|    487.97|
|I0000007| O000002|      Expert School|       Sports|       8|    476.36|
|I0000008| O000002|    Prevent Million|       Sports|       1|    198.84|
|I0000009| O000002|           Cup Full|Home & Garden|      10|    274.63|
|I0000010| O000003|        After Civil|        Books|       1|    182.89|
|I0000011| O000003|         Appear Nor

In [33]:
df_order_items.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: float (nullable = true)



In [44]:
orderStruct = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", FloatType(), True),
    StructField("discount_pct",IntegerType(), True)
    ])

In [45]:
df_orders = spark.read.format('csv')\
                .schema(orderStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/orders.csv')
df_orders.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
| O000001|     C00078|03/08/2022|cancelled|      230.07|           5|
| O000002|     C00044|22/01/2021|completed|      221.77|           5|
| O000003|     C00286|2024-10-17|  pending|      1795.1|          15|
| O000004|     C00398|16/05/2022| refunded|      404.43|          20|
| O000005|     C00231|21/03/2024|cancelled|      -26.78|          15|
| O000006|     C00302|18/07/2024|cancelled|      123.89|          15|
| O000007|     C00292|2023-04-05|completed|     1677.58|          20|
| O000008|     C00313|24/10/2021|completed|     1492.92|          25|
| O000009|     C00391|2021-01-28|  pending|      394.15|           5|
| O000010|     C00136|18/11/2023|completed|      118.46|          25|
| O000011|     C00081|2021-01-07|  pending|      354.52|           5|
| O000012|     C0028

In [38]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- discount_pct: integer (nullable = true)



In [42]:
returnStruct = StructType ([
    StructField('return_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('return_date', StringType(), True),
    StructField('reason', StringType(), True),
    StructField('refund_amount', FloatType(), True)
])

In [43]:
df_returns = spark.read.format('csv')\
                .schema(returnStruct)\
                .option('inferschema', False)\
                .option('header', True)\
                .load('C:/Users/ADMIN/Desktop/Lux Dev Data Enginering/Assignment_1/data/returns.csv')
df_returns.show()

+---------+--------+-----------+-------------------+-------------+
|return_id|order_id|return_date|             reason|refund_amount|
+---------+--------+-----------+-------------------+-------------+
|  R000001| O001713| 19/04/2024|       changed mind|       349.04|
|  R000002| O001177| 05/08/2022|         wrong item|      1721.96|
|  R000003| O000991| 2021-01-08|   not as described|        27.77|
|  R000004| O000017| 12/01/2023|          defective|       260.67|
|  R000005| O001787| 2021-09-05|          defective|       754.97|
|  R000006| O000732| 20/06/2022|          defective|       240.01|
|  R000007| O001191| 05/06/2021|   not as described|       913.52|
|  R000008| O000666| 2023-01-02|       changed mind|        27.48|
|  R000009| O000147| 2023-05-05|          defective|         71.9|
|  R000010| O000746| 2023-12-03|         wrong item|       165.16|
|  R000011| O001509| 2022-09-05|         wrong item|       492.46|
|  R000012| O001431| 22/12/2023|damaged in shipping|      1730

In [41]:
df_returns.printSchema()

root
 |-- return_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- return_date: date (nullable = true)
 |-- reason: string (nullable = true)
 |-- refund_amount: float (nullable = true)



note: while designing the structtypes for date columns, since the dates have a different order, they are interpreted as null thus string type was the option

Task 02: Clean in the exact sequence the spec says deduplicate-> normalize dates -> lowercase tier -> drop nulls -> flag negatives. Handling the mixed date formats(year-month-day vs day-month-year). use F.to/-date with coalesce to try both formats.

In [70]:
df_customers = df_customers.dropDuplicates().show()

+-----------+-------------------+--------------------+-------+-------------+-----------+
|customer_id|      customer_name|               email|country|customer_tier|signup_date|
+-----------+-------------------+--------------------+-------+-------------+-----------+
|     C00059|      Don Tucker MD| dramsey@example.org|     VU|       Bronze| 2019-11-17|
|     C00173|       Eric Sanders|   wking@example.net|     ET|       SILVER| 2018-05-11|
|     C00174|      Richard Glenn|joshua56@example.com|     CH|         Gold| 07/06/2020|
|     C00076| Dr. Elizabeth Ward|alexis82@example.com|     BD|         GOLD| 2020-05-29|
|     C00147|    Victoria Morris|edward17@example.com|     ZM|       silver| 2022-10-20|
|     C00181|     Kendra Sanders|  john10@example.net|     ER|       BRONZE| 27/08/2018|
|     C00151|Mrs. Maria Williams| hritter@example.net|     VA|       SILVER| 03/12/2020|
|     C00093|       Adam Russell|victoriadurham@ex...|     CL|       SILVER| 2019-05-08|
|     C00092|        

In [69]:
df_order_items = df_order_items.dropDuplicates().show()

+--------+--------+--------------------+-------------+--------+----------+
| item_id|order_id|        product_name|     category|quantity|unit_price|
+--------+--------+--------------------+-------------+--------+----------+
|I0000054| O000014|         Fire Decide|         Toys|       1|     364.0|
|I0000132| O000041|      Almost Defense|       Beauty|       6|    462.74|
|I0000544| O000171|  Region Participant|       Beauty|       6|    413.75|
|I0000604| O000191|   Public Difference|       Beauty|       7|    210.44|
|I0001012| O000333|           War Bring|Home & Garden|       5|    499.49|
|I0001048| O000345|         Church Read|       Sports|      10|    217.79|
|I0001284| O000424|          Edge Raise|       Sports|       2|    110.43|
|I0001300| O000428|       Product Avoid|         Toys|       9|    459.94|
|I0001409| O000465|     Create Describe|       Sports|       2|    329.54|
|I0001430| O000471|      Reflect Choice|     Clothing|       5|    329.96|
|I0001448| O000478|      

In [68]:
df_returns = df_returns.dropDuplicates().show()

+---------+--------+-----------+-------------------+-------------+
|return_id|order_id|return_date|             reason|refund_amount|
+---------+--------+-----------+-------------------+-------------+
|  R000022| O001233| 2024-03-31|   not as described|      1253.68|
|  R000058| O000695| 17/07/2022|       changed mind|       163.36|
|  R000107| O001953| 07/06/2021|       changed mind|       308.61|
|  R000154| O001746| 06/04/2023|damaged in shipping|       857.19|
|  R000115| O000744| 2022-05-11|   not as described|       963.53|
|  R000178| O000184| 2021-03-03|         wrong item|       264.45|
|  R000160| O000002| 2021-08-22|   not as described|       191.74|
|  R000162| O000093| 08/06/2022|damaged in shipping|       665.15|
|  R000070| O001731| 2021-12-24|         wrong item|       645.92|
|  R000002| O001177| 05/08/2022|         wrong item|      1721.96|
|  R000030| O000231| 2021-07-27|       changed mind|       472.42|
|  R000133| O001808| 2024-02-27|       changed mind|       445

In [67]:
df_orders = df_orders.dropDuplicates().show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
| O000398|     C00066|2022-06-15|  pending|      736.44|          20|
| O000531|     C00355|2023-02-22|completed|       19.92|           5|
| O000664|     C00257|2024-08-07|cancelled|      605.83|          25|
| O000727|     C00055|06/11/2023|completed|      353.01|           0|
| O000916|     C00073|09/12/2021|cancelled|      463.34|          25|
| O001307|     C00350|2024-06-03|  pending|      650.34|           0|
| O001930|     C00107|2022-11-18|cancelled|      569.28|          20|
| O001933|     C00193|2022-10-03|  pending|      899.03|          25|
| O000072|     C00254|22/09/2023|  pending|      668.16|           0|
| O000167|     C00235|2024-11-06|completed|     1781.63|          25|
| O000442|     C00136|11/11/2023|cancelled|      597.83|          25|
| O000521|     C0015

In [66]:
df_orders.withColumn('formatted_order_date', date_format('order_date', 'yyyy-MM-dd')).show()

{"ts": "2026-06-20 15:06:03.514", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '03/08/2022' of the type \"STRING\" cannot be cast to \"TIMESTAMP\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "date_format", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o188.showString.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '03/08/2022' of the type \"STRING\" cannot be cast to \"TIMESTAMP\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"date_format\" 

DateTimeException: [CAST_INVALID_INPUT] The value '03/08/2022' of the type "STRING" cannot be cast to "TIMESTAMP" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"date_format" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
